# 01 — Fashion-MNIST: tải và kiểm tra dataset

Notebook Julia để tải dataset chính thức, kiểm tra checksum, đọc IDX, gộp 70.000 ảnh và lưu dữ liệu/kết quả trong repository trên Google Drive.

Repo nằm tại `/content/gdrive/MyDrive/[DM]-FIM`. Trước khi chạy, mount Google Drive vào `/content/gdrive`; notebook sẽ xác minh repo và kích hoạt `Project.toml` ngay tại đó. Chạy bằng Julia ≥ 1.9.

Nguồn: [Fashion-MNIST của Zalando](https://github.com/zalandoresearch/fashion-mnist), cũng được ghi trong `docs/plan.md`.

## 1. Mount Drive và kích hoạt project trong repository

Julia đọc source và cấu hình từ repo trên Drive. Archive raw, PNG gộp và báo cáo cũng được lưu trong các thư mục `data/` và `results/` của chính repo.

In [ ]:
using Pkg

# Mountpoint và đường dẫn repo đã được người dùng xác định.
drive_mount = "/content/gdrive"
isdir(joinpath(drive_mount, "MyDrive")) || error(
    "Chưa mount Google Drive tại /content/gdrive. Hãy mount Drive rồi chạy lại."
)
repo_root = get(ENV, "FASHION_MNIST_REPO_ROOT", "/content/gdrive/MyDrive/[DM]-FIM")

isfile(joinpath(repo_root, "Project.toml")) || error(
    "Không tìm thấy Project.toml tại repo root: $repo_root"
)
isfile(joinpath(repo_root, "docs", "plan.md")) || error(
    "Repo root không có docs/plan.md: $repo_root"
)
isdir(joinpath(repo_root, "src")) || error(
    "Repo root không có thư mục src/: $repo_root"
)

# Công cụ ghi PNG thuộc môi trường notebook, không thuộc runtime TC.
notebook_env = joinpath(repo_root, "environments", "fashion_notebooks")
Pkg.activate(notebook_env)
cd(notebook_env) do
    Pkg.develop(path="../..")
end
Pkg.instantiate()

using EclatFIM
using SHA: sha256
using TOML

const data_dir = joinpath(repo_root, "data", "raw", "fashion-mnist")
const results_dir = joinpath(repo_root, "results", "fashion")
const COMBINED_DIR = joinpath(repo_root, "data", "processed", "fashion-mnist", "combined")

mkpath(data_dir)
mkpath(results_dir)

@assert VERSION >= v"1.9"
println("Julia ", VERSION)
println("Project: ", Base.active_project())
println("Repository: ", repo_root)
println("Raw data: ", data_dir)
println("Results: ", results_dir)

## 2. Tải bốn archive lên Google Drive và kiểm tra SHA-256

Các file `.gz` được tải trực tiếp vào `data/raw/fashion-mnist/` bên trong repo trên Drive. Mỗi file được đối chiếu checksum cố định trong source trước khi đọc; file tải lỗi hoặc sai nội dung sẽ bị từ chối.

In [ ]:
archive_paths = download_fashion_mnist(data_dir)
expected_files = EclatFIM.FASHION_FILES

for (archive, expected) in zip(archive_paths, expected_files)
    actual = bytes2hex(sha256(read(archive)))
    @assert actual == expected.sha256
    println("SHA-256 OK  ", expected.name, "  (", filesize(archive), " bytes)")
end
println("Đã xác minh đủ 4 archive từ ", EclatFIM.FASHION_SOURCE)


## 3. Đọc ảnh và nhãn IDX

IDX lưu ảnh theo thứ tự dòng. Hàm đọc chuyển đúng về mảng Julia `images[row, column, image]`, không xoay ảnh; cường độ pixel vẫn là `UInt8` trong khoảng 0–255. Nhãn lưu tách khỏi ảnh.

In [ ]:
train = load_fashion_mnist(data_dir; split=:train)
test = load_fashion_mnist(data_dir; split=:test)

println("Train: ", size(train.images), " ảnh; ", length(train.labels), " nhãn")
println("Test:  ", size(test.images), " ảnh; ", length(test.labels), " nhãn")
println("Kiểu pixel: ", eltype(train.images), "; giá trị train: ", extrema(train.images))


## 4. Xác nhận shape, nhãn và train/test split

Các assertion xác nhận đủ 60.000 ảnh train và 10.000 ảnh test; hai split có ảnh 28×28, nhãn 0–9 và 6.000/1.000 ảnh mỗi lớp.

In [ ]:
@assert size(train.images) == (28, 28, 60_000)
@assert size(test.images) == (28, 28, 10_000)
@assert eltype(train.images) == UInt8 && eltype(test.images) == UInt8
@assert length(train.labels) == size(train.images, 3)
@assert length(test.labels) == size(test.images, 3)
@assert all(label -> 0 <= label <= 9, train.labels)
@assert all(label -> 0 <= label <= 9, test.labels)
@assert sort(unique(train.labels)) == collect(0:9)
@assert sort(unique(test.labels)) == collect(0:9)

train_counts = [count(==(label), train.labels) for label in 0:9]
test_counts = [count(==(label), test.labels) for label in 0:9]
@assert all(==(6_000), train_counts)
@assert all(==(1_000), test_counts)

println("PASS: đúng shape, grayscale UInt8, nhãn 0–9 và split train/test.")
for label in 0:9
    println(rpad("$(label) $(FASHION_CLASSES[label + 1])", 22),
        "train=", rpad(string(train_counts[label + 1]), 7),
        "test=", test_counts[label + 1])
end


## 5. Gộp train/test, xuất PNG trên ổ local và đóng gói lên Drive

EDA-1 khảo sát toàn bộ **70.000 ảnh**, giữ train trước rồi test sau. PNG được tạo và kiểm tra trên ổ local Colab (`/content`) để tránh ghi 70.000 file nhỏ trực tiếp lên Drive. Sau khi kiểm tra đủ PNG, cell đóng gói `images/`, `image_index.csv` và `metadata.toml` thành một file TAR không nén rồi chép archive đó lên Drive.

`dataset.images` và `dataset.labels` là đầu vào EDA-1; `source_split` và `source_index` cho phép truy ngược từng ảnh. Bên trong archive, CSV index dùng `image_id` bắt đầu từ 1 và lưu đường dẫn tương đối `combined/images/...`.

In [ ]:
n_train = size(train.images, 3)
n_test = size(test.images, 3)

dataset = (
    images=cat(train.images, test.images; dims=3),
    labels=vcat(train.labels, test.labels),
    split=:combined,
    source_split=vcat(fill(:train, n_train), fill(:test, n_test)),
    source_index=vcat(collect(1:n_train), collect(1:n_test)),
)

# Tên ngắn dùng trong plan EDA-1: X[row, column, image_id], y[image_id].
X = dataset.images
y = dataset.labels
combined_counts = [count(==(label), y) for label in 0:9]

@assert size(X) == (28, 28, 70_000)
@assert eltype(X) == UInt8
@assert length(y) == length(dataset.source_split) == length(dataset.source_index) == 70_000
@assert all(==(7_000), combined_counts)
@assert view(X, :, :, 1:n_train) == train.images
@assert view(X, :, :, n_train + 1:n_train + n_test) == test.images
@assert y[1:n_train] == train.labels && y[n_train + 1:end] == test.labels
@assert dataset.source_split[1] == :train && dataset.source_split[end] == :test
@assert dataset.source_index[n_train] == n_train && dataset.source_index[n_train + 1] == 1

println("Combined dataset: ", size(X), "; labels: ", length(y))
println("Mỗi lớp: ", combined_counts)
println("Thứ tự image_id: 1:60_000 từ train; 60_001:70_000 từ test.")

# Tạo PNG trên ổ local của Colab; chỉ archive cuối cùng mới được chép lên Drive.
using PNGFiles
using ColorTypes: Gray
using FixedPointNumbers: N0f8

local_work_dir = "/content/fashion-mnist"
local_combined_dir = joinpath(local_work_dir, "combined")
images_dir = joinpath(local_combined_dir, "images")
mkpath(images_dir)
index_path = joinpath(local_combined_dir, "image_index.csv")

# Nơi lưu một archive duy nhất trên Drive.
combined_dir = COMBINED_DIR
mkpath(combined_dir)
archive_path = joinpath(combined_dir, "fashion-mnist-combined.tar")

function export_combined_pngs(images_dir, index_path, dataset)
    temporary_index = index_path * ".tmp"
    open(temporary_index, "w") do io
        println(io, "image_id,relative_path,label,source_split,source_index,sha256")
        for image_id in eachindex(dataset.labels)
            filename = lpad(string(image_id), 6, '0') * ".png"
            path = joinpath(images_dir, filename)
            image = @view dataset.images[:, :, image_id]

            # Ghi pixel UInt8 gốc, không chuẩn hóa hoặc resize.
            PNGFiles.save(path, Gray.(reinterpret(N0f8, image)))

            # Đọc lại PNG vừa ghi và xác minh mọi pixel trước khi ghi index.
            decoded = PNGFiles.load(path)
            @assert size(decoded) == (28, 28)
            @assert eltype(decoded) == Gray{N0f8}
            @assert reinterpret(UInt8, decoded) == image

            checksum = bytes2hex(open(sha256, path))
            println(io, "$image_id,combined/images/$filename,$(dataset.labels[image_id]),$(dataset.source_split[image_id]),$(dataset.source_index[image_id]),$checksum")

            if image_id % 10_000 == 0
                println("Đã lưu và kiểm tra ", image_id, " / ", length(dataset.labels), " ảnh PNG trên /content")
            end
        end
    end
    mv(temporary_index, index_path; force=true)
end

export_combined_pngs(images_dir, index_path, dataset)
@assert count(endswith(".png"), readdir(images_dir)) == 70_000
index_rows = readlines(index_path)
@assert length(index_rows) == 70_001
for image_id in eachindex(y)
    fields = split(index_rows[image_id + 1], ',')
    @assert parse(Int, fields[1]) == image_id
    @assert fields[2] == "combined/images/" * lpad(string(image_id), 6, '0') * ".png"
    @assert parse(Int, fields[3]) == y[image_id]
    @assert fields[4] == string(dataset.source_split[image_id])
    @assert parse(Int, fields[5]) == dataset.source_index[image_id]
end

combined_metadata = Dict(
    "dataset" => "Fashion-MNIST combined",
    "source" => EclatFIM.FASHION_SOURCE,
    "format" => "individual PNG files, grayscale 8-bit, inside an uncompressed TAR archive",
    "archive" => "fashion-mnist-combined.tar",
    "images" => length(y),
    "image_shape" => [28, 28],
    "pixel_type" => "UInt8",
    "class_counts" => combined_counts,
    "original_train_images" => n_train,
    "original_test_images" => n_test,
    "order" => "train first, then test; original order retained",
    "index_base" => 1,
    "roundtrip_verified" => true,
    "verified_png_images" => length(y),
    "sha256" => Dict("image_index.csv" => bytes2hex(open(sha256, index_path))),
    "png_checksums" => "sha256 column in combined/image_index.csv",
    "source_archives" => [Dict("filename" => file.name, "sha256" => file.sha256)
        for file in expected_files],
)
metadata_path = joinpath(local_combined_dir, "metadata.toml")
open(metadata_path * ".tmp", "w") do io
    TOML.print(io, combined_metadata; sorted=true)
end
mv(metadata_path * ".tmp", metadata_path; force=true)

# PNG đã được nén nội bộ; tạo TAR không nén để đóng gói nhanh, giữ một file trên Drive.
local_archive_path = joinpath(local_work_dir, "fashion-mnist-combined.tar")
run(`tar -cf $local_archive_path -C $local_work_dir combined`)
run(pipeline(`tar -tf $local_archive_path`, stdout=devnull))
archive_sha256 = bytes2hex(open(sha256, local_archive_path))

# Chép archive tạm rồi đổi tên sau khi checksum khớp để không làm hỏng bản cũ.
archive_temporary_path = archive_path * ".tmp"
cp(local_archive_path, archive_temporary_path; force=true)
@assert bytes2hex(open(sha256, archive_temporary_path)) == archive_sha256
mv(archive_temporary_path, archive_path; force=true)

# Xóa IDX combined cũ nếu tồn tại, sau khi archive PNG đã lưu và xác minh xong.
for filename in ("images-idx3-ubyte.gz", "labels-idx1-ubyte.gz")
    old_path = joinpath(combined_dir, filename)
    isfile(old_path) && rm(old_path)
end

archive_relpath = relpath(archive_path, repo_root)
println("Archive trên Drive: ", archive_relpath)
println("Kích thước archive: ", filesize(archive_path), " bytes")
println("SHA-256 archive: ", archive_sha256)
println("Index bên trong archive: combined/image_index.csv")


## 6. Lưu báo cáo xác minh lên Google Drive

Báo cáo ghi đường dẫn và SHA-256 của archive trên Drive. Archive chứa 70.000 PNG đã round-trip kiểm tra, `image_index.csv` và `metadata.toml`.

In [ ]:
mkpath(results_dir)
report = Dict(
    "dataset" => "Fashion-MNIST",
    "source" => EclatFIM.FASHION_SOURCE,
    "julia_version" => string(VERSION),
    "image_shape" => [28, 28],
    "pixel_type" => string(eltype(train.images)),
    "checks_passed" => true,
    "eda_scope" => "combined_train_test",
    "combined_images" => size(X, 3),
    "combined_folder" => relpath(combined_dir, repo_root),
    "combined_archive" => archive_relpath,
    "combined_archive_sha256" => archive_sha256,
    "combined_format" => "individual PNG grayscale 8-bit inside uncompressed TAR",
    "combined_index" => archive_relpath * "#combined/image_index.csv",
    "combined_roundtrip_verified" => true,
    "combined_class_counts" => combined_counts,
    "combined_pixel_count" => length(X),
    "combined_order" => "train first, then test; original order retained",
    "provenance" => "combined/image_index.csv inside archive: image_id, relative_path, label, source_split, source_index, sha256",
    "train_images" => size(train.images, 3),
    "test_images" => size(test.images, 3),
    "train_class_counts" => train_counts,
    "test_class_counts" => test_counts,
    "archives" => [Dict("filename" => file.name, "sha256" => file.sha256)
        for file in expected_files],
)
report_path = joinpath(results_dir, "dataset_validation.toml")
open(report_path, "w") do io
    TOML.print(io, report; sorted=true)
end
println("Báo cáo: ", relpath(report_path, repo_root))


Archive và báo cáo được lưu trong repository trên Drive:

```text
/content/gdrive/MyDrive/[DM]-FIM/
├── data/
│   ├── raw/fashion-mnist/                 # bốn archive .gz
│   └── processed/fashion-mnist/combined/
│       └── fashion-mnist-combined.tar     # combined/images/*.png, image_index.csv, metadata.toml
└── results/fashion/dataset_validation.toml
```

TAR không nén thêm các PNG. Để dùng lại trên Colab, chép archive từ Drive về `/content`, giải nén tại đó rồi đọc `combined/image_index.csv` theo `image_id`:

```julia
local_archive = joinpath("/content", "fashion-mnist-combined.tar")
run(`cp $archive_path $local_archive`)
run(`tar -xf $local_archive -C /content`)
image = PNGFiles.load("/content/combined/images/000001.png")
pixels = reinterpret(UInt8, image)  # 28×28, đúng cường độ 0–255 gốc
```